In [22]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler
from scipy.spatial.distance import pdist, squareform
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.manifold import MDS
from sklearn.decomposition import PCA
import colorsys
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from db_connection import db, Songs, SongAudioFeatures

In [23]:
db.connect(reuse_if_open=True)
query = (
    SongAudioFeatures
    .select(SongAudioFeatures, Songs)
    .join(Songs, on=(SongAudioFeatures.song == Songs.id))
)

audio_rows = []
lyrics_emb_rows = []
image_emb_rows = []
song_ids = []

for row in query:
    song = row.song  # the joined Songs instance
    sid = str(song.id)
    song_ids.append(sid)
    audio_record = {col: getattr(row, col, None) for col in [
    "acousticness",
    "danceability",
    "energy",
    "instrumentalness",
    "key",
    "liveness",
    "loudness",
    "mode",
    "speechiness",
    "tempo",
    "time_signature",
    "valence",
    "num_samples",
    "duration",
    "duration_ms",
    "end_of_fade_in",
    "start_of_fade_out",
    "tempo_confidence",
    "time_signature_confidence",
    "key_confidence",
    "mode_confidence",
    ]}
    audio_record["song_duration_ms"] = song.duration_ms
    audio_record["popularity"] = song.popularity
    audio_record["explicit"] = int(song.explicit) if song.explicit is not None else None
    audio_rows.append(audio_record)
    if row.lyrics_embeddings is not None:
        lyrics_emb_rows.append(list(row.lyrics_embeddings))
    else:
        lyrics_emb_rows.append([np.nan] * 768)
    if row.image_embeddings is not None:
        image_emb_rows.append(list(row.image_embeddings))
    else:
        image_emb_rows.append([np.nan] * 768)
df_audio = pd.DataFrame(audio_rows, index=song_ids)
df_audio.index.name = "song_id"

lyrics_emb_columns = [f"lyrics_emb_{i}" for i in range(768)]
df_lyrics_emb = pd.DataFrame(lyrics_emb_rows, index=song_ids, columns=lyrics_emb_columns)
df_lyrics_emb.index.name = "song_id"

image_emb_columns = [f"image_emb_{i}" for i in range(768)]
df_image_emb = pd.DataFrame(image_emb_rows, index=song_ids, columns=image_emb_columns)
df_image_emb.index.name = "song_id"
db.close()

True

In [24]:
# filter out songs with missing audio features
df_audio = df_audio.dropna(subset=["acousticness", "danceability", "energy", "instrumentalness", "key",
                                   "liveness", "loudness", "mode", "speechiness", "tempo", "time_signature", "valence"])
df_lyrics_emb = df_lyrics_emb.dropna()
df_image_emb = df_image_emb.dropna()

common_song_ids = set(df_audio.index) & set(df_lyrics_emb.index) & set(df_image_emb.index)
common_song_ids = list(common_song_ids)  # convert set to list
df_audio = df_audio.loc[common_song_ids]
df_lyrics_emb = df_lyrics_emb.loc[common_song_ids]
df_image_emb = df_image_emb.loc[common_song_ids]

In [25]:
df_audio.to_csv("song_audio_features.csv")
df_lyrics_emb.to_csv("song_lyrics_embeddings.csv")
df_image_emb.to_csv("song_image_embeddings.csv")

In [26]:
def euclidean_distance_matrix(df: pd.DataFrame) -> pd.DataFrame:
    filled = df.fillna(df.median())
    dist_vec = pdist(filled.values, metric="euclidean")
    dist_mat = squareform(dist_vec)
    np.fill_diagonal(dist_mat, np.nan)
    return pd.DataFrame(dist_mat, index=df.index, columns=df.index)
def normalize_df(df: pd.DataFrame) -> pd.DataFrame:
    scaler = MinMaxScaler()
    scaled = scaler.fit_transform(df)
    return pd.DataFrame(scaled, index=df.index, columns=df.columns)


In [27]:
df_audio_norm = normalize_df(df_audio)
df_lyrics_norm = normalize_df(df_lyrics_emb)
df_image_norm = normalize_df(df_image_emb)
dist_audio = euclidean_distance_matrix(df_audio_norm)
dist_lyrics = euclidean_distance_matrix(df_lyrics_norm)
dist_image = euclidean_distance_matrix(df_image_norm)
for label, raw, normed, dist in [
        ("Audio Features", df_audio, df_audio_norm, dist_audio),
        ("Lyrics Embeddings", df_lyrics_emb, df_lyrics_norm, dist_lyrics),
        ("Image Embeddings", df_image_emb, df_image_norm, dist_image),
    ]:
    print(f"=== {label} ===")
    print(f"  Raw shape:        {raw.shape}")
    print(f"  Normalized range: [{normed.min().min():.4f}, {normed.max().max():.4f}]")
    print(f"  Distance matrix:  {dist.shape}")
    print(f"  Mean distance:    {dist.values[np.triu_indices_from(dist.values, k=1)].mean():.4f}")
    print()

=== Audio Features ===
  Raw shape:        (427, 24)
  Normalized range: [0.0000, 1.0000]
  Distance matrix:  (427, 427)
  Mean distance:    1.3848

=== Lyrics Embeddings ===
  Raw shape:        (427, 768)
  Normalized range: [0.0000, 1.0000]
  Distance matrix:  (427, 427)
  Mean distance:    4.8536

=== Image Embeddings ===
  Raw shape:        (427, 768)
  Normalized range: [0.0000, 1.0000]
  Distance matrix:  (427, 427)
  Mean distance:    7.2734



In [28]:
dist_audio.to_csv("song_audio_distance_matrix.csv")
dist_lyrics.to_csv("song_lyrics_distance_matrix.csv")
dist_image.to_csv("song_image_distance_matrix.csv")

In [29]:
def find_optimal_k(df: pd.DataFrame, k_range: range = range(2, 11), random_state: int = 42) -> int:
    """Find the optimal number of clusters using silhouette score."""
    filled = df.fillna(df.median())
    best_k = k_range.start
    best_score = -1
    for k in k_range:
        km = KMeans(n_clusters=k, n_init=10, random_state=random_state)
        labels = km.fit_predict(filled.values)
        score = silhouette_score(filled.values, labels)
        print(f"  k={k}: silhouette={score:.4f}")
        if score > best_score:
            best_score = score
            best_k = k
    print(f"  → Best k={best_k} (silhouette={best_score:.4f})")
    return best_k

def kmeans_cluster(df: pd.DataFrame, n_clusters: int, random_state: int = 42) -> pd.Series:
    filled = df.fillna(df.median())
    km = KMeans(n_clusters=n_clusters, n_init=10, random_state=random_state)
    labels = km.fit_predict(filled.values)
    return pd.Series(labels, index=df.index, name="cluster"), km

In [30]:
print("Audio features:")
k_audio = find_optimal_k(df_audio_norm)
print("\nLyrics embeddings:")
k_lyrics = find_optimal_k(df_lyrics_norm)
print("\nImage embeddings:")
k_image = find_optimal_k(df_image_norm)

audio_clusters, audio_km = kmeans_cluster(df_audio_norm, n_clusters=k_audio)
lyrics_clusters, lyrics_km = kmeans_cluster(df_lyrics_norm, n_clusters=k_lyrics)
image_clusters, image_km = kmeans_cluster(df_image_norm, n_clusters=k_image)

Audio features:
  k=2: silhouette=0.2686
  k=3: silhouette=0.2715
  k=4: silhouette=0.1992
  k=5: silhouette=0.1358
  k=6: silhouette=0.1884
  k=7: silhouette=0.1543
  k=8: silhouette=0.1504
  k=9: silhouette=0.1300
  k=10: silhouette=0.1459
  → Best k=3 (silhouette=0.2715)

Lyrics embeddings:


  k=2: silhouette=0.3108
  k=3: silhouette=0.3496
  k=4: silhouette=0.3462
  k=5: silhouette=0.3471
  k=6: silhouette=0.3473
  k=7: silhouette=0.1353
  k=8: silhouette=0.1304
  k=9: silhouette=0.1072
  k=10: silhouette=0.1052
  → Best k=3 (silhouette=0.3496)

Image embeddings:
  k=2: silhouette=0.1378
  k=3: silhouette=0.1503
  k=4: silhouette=0.2043
  k=5: silhouette=0.2452
  k=6: silhouette=0.2455
  k=7: silhouette=0.3003
  k=8: silhouette=0.3417
  k=9: silhouette=0.3838
  k=10: silhouette=0.4160
  → Best k=10 (silhouette=0.4160)


In [31]:
def _generate_distinct_colors(n: int, seed: int = 42) -> list[str]:
    """
    Generate *n* visually distinct colours by spacing hues evenly around
    the HSL wheel with a random offset, then jittering saturation/lightness.
    """
    rng = np.random.RandomState(seed)
    offset = rng.uniform(0, 1)
    colors = []
    for i in range(n):
        hue = (i / n + offset) % 1.0
        sat = rng.uniform(0.55, 0.85)
        lit = rng.uniform(0.45, 0.62)
        r, g, b = colorsys.hls_to_rgb(hue, lit, sat)
        colors.append(f"#{int(r*255):02x}{int(g*255):02x}{int(b*255):02x}")
    return colors


def _draw_cluster_hull(ax: plt.Axes, points: np.ndarray, color: str,
                       alpha_fill: float = 0.08, alpha_edge: float = 0.45):
    """Draw a convex-hull boundary around a set of 2-D points."""
    if len(points) < 3:
        return
    try:
        hull = ConvexHull(points)
    except Exception:
        return
    vertices = np.append(hull.vertices, hull.vertices[0])  # close the polygon
    ax.fill(
        points[vertices, 0], points[vertices, 1],
        color=color, alpha=alpha_fill,
    )
    ax.plot(
        points[vertices, 0], points[vertices, 1],
        color=color, alpha=alpha_edge, linewidth=1.8, linestyle="--",
    )


def plot_clusters_2d(
    df_norm: pd.DataFrame,
    clusters: pd.Series,
    km: KMeans,
    title: str,
    ax: plt.Axes,
    color_seed: int = 42,
):
    """
    Reduce *df_norm* to 2-D with PCA, scatter-plot coloured by cluster,
    draw convex-hull borders, and mark centroids with ×.
    """
    filled = df_norm.fillna(df_norm.median())
    pca = PCA(n_components=2)
    coords = pca.fit_transform(filled.values)
    var = pca.explained_variance_ratio_

    centres_2d = pca.transform(km.cluster_centers_)
    colors = _generate_distinct_colors(km.n_clusters, seed=color_seed)

    # ── per-cluster: hull + scatter ──────────────────────────────
    for k in range(km.n_clusters):
        mask = clusters.values == k
        pts = coords[mask]
        c = colors[k]

        # convex-hull boundary
        _draw_cluster_hull(ax, pts, color=c)

        # data points
        ax.scatter(
            pts[:, 0], pts[:, 1],
            c=c, label=f"Cluster {k}",
            s=36, alpha=0.80,
            edgecolors="white", linewidths=0.5,
            zorder=3,
        )

    # ── centroids ────────────────────────────────────────────────
    ax.scatter(
        centres_2d[:, 0], centres_2d[:, 1],
        c="black", marker="X", s=160,
        edgecolors="white", linewidths=1.2,
        zorder=6, label="Centroid",
    )

    # ── axes polish ──────────────────────────────────────────────
    ax.set_xlabel(f"PC 1  ({var[0]*100:.1f}% var)", fontsize=10)
    ax.set_ylabel(f"PC 2  ({var[1]*100:.1f}% var)", fontsize=10)
    ax.set_title(title, fontsize=13, fontweight="bold", pad=10)
    ax.legend(
        fontsize=8, loc="best", framealpha=0.9,
        edgecolor="#cccccc", fancybox=True,
    )
    ax.grid(True, linewidth=0.3, alpha=0.4)
    ax.set_axisbelow(True)
    ax.spines[["top", "right"]].set_visible(False)


In [32]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
plot_clusters_2d(df_audio_norm, audio_clusters, audio_km,
                     f"Audio Features (k={k_audio})", axes[0])
plot_clusters_2d(df_lyrics_norm, lyrics_clusters, lyrics_km,
                     f"Lyrics Embeddings (k={k_lyrics})", axes[1])
plot_clusters_2d(df_image_norm, image_clusters, image_km,
                     f"Album Art Embeddings (k={k_image})", axes[2])
fig.suptitle("K-Means (dynamic k via silhouette) — PCA 2-D Projection", fontsize=14, y=1.02)
fig.tight_layout()
fig.savefig("cluster_plots.png", dpi=150, bbox_inches="tight")

In [33]:
def merge_distance_matrices(
    matrices: list[pd.DataFrame],
    weights: list[float] | None = None,
) -> pd.DataFrame:
    if weights is None:
        weights = [1.0] * len(matrices)
    weights = np.array(weights) / np.sum(weights)       # normalise to 1

    scaled = []
    for mat in matrices:
        vals = mat.values
        lo, hi = vals.min(), vals.max()
        if hi - lo > 0:
            scaled.append((vals - lo) / (hi - lo))
        else:
            scaled.append(np.zeros_like(vals))

    merged = sum(w * s for w, s in zip(weights, scaled))
    return pd.DataFrame(merged, index=matrices[0].index,
                        columns=matrices[0].columns)


In [34]:
def merge_features(
    dfs: list[pd.DataFrame],
    weights: list[float] | None = None,
) -> pd.DataFrame:
    """
    Concatenate multiple normalized feature DataFrames side-by-side.
    Optionally scale each block by a weight so you can emphasise
    one feature space over another.

    The result is a (n_songs × total_features) DataFrame that works
    directly with PCA + KMeans.
    """
    if weights is None:
        weights = [1.0] * len(dfs)
    weighted = [df * w for df, w in zip(dfs, weights)]
    return pd.concat(weighted, axis=1)


In [35]:
dist_merged = merge_distance_matrices(
    [dist_audio, dist_lyrics, dist_image],
    weights=[1.0, 1.0, 0.2],   # tune these
)
merged_features = merge_features(
    [df_audio_norm, df_lyrics_norm, df_image_norm],
    weights=[1.0, 1.0, 0.2],
)
print("Merged features:")
k_merged = find_optimal_k(merged_features)
merged_clusters, merged_km = kmeans_cluster(merged_features, n_clusters=k_merged)

Merged features:


  k=2: silhouette=0.2600
  k=3: silhouette=0.2885
  k=4: silhouette=0.2797
  k=5: silhouette=0.2788
  k=6: silhouette=0.2358
  k=7: silhouette=0.0736
  k=8: silhouette=0.0779
  k=9: silhouette=0.0739
  k=10: silhouette=0.0766
  → Best k=3 (silhouette=0.2885)


In [36]:
def save_clusters_2d_plot(df_norm, clusters, km, title, filepath):
    fig, ax = plt.subplots(figsize=(8, 6))
    plot_clusters_2d(df_norm, clusters, km, title, ax)
    fig.savefig(filepath, dpi=150, bbox_inches="tight")
    plt.close(fig)

save_clusters_2d_plot(df_audio_norm, audio_clusters, audio_km, "Audio Features", "audio_features.png")
save_clusters_2d_plot(df_lyrics_norm, lyrics_clusters, lyrics_km, "Lyrics Embeddings", "lyrics_embeddings.png")
save_clusters_2d_plot(df_image_norm, image_clusters, image_km, "Album Art Embeddings", "album_art_embeddings.png")
save_clusters_2d_plot(merged_features, merged_clusters, merged_km, "All Features", "all_features.png")

In [37]:
# ── Key / Tempo flow distance ─────────────────────────────────────────
# Musical key distance via circle-of-fifths (accounts for harmonic compatibility).
# Mode is folded in: relative major↔minor costs less than distant keys.

FIFTHS_POS = {0:0, 7:1, 2:2, 9:3, 4:4, 11:5, 6:6, 1:7, 8:8, 3:9, 10:10, 5:11}

def key_distance(k1, m1, k2, m2):
    """Distance between two keys on the circle of fifths (0-1 normalised).
    Relative major/minor pairs (3 semitones apart) get a small bonus."""
    p1, p2 = FIFTHS_POS[int(k1)], FIFTHS_POS[int(k2)]
    raw = abs(p1 - p2)
    d = min(raw, 12 - raw) / 6.0  # max distance on circle = 6
    # relative major/minor bonus: if modes differ and keys are 3 semitones apart
    if m1 != m2 and abs(int(k1) - int(k2)) % 12 in (3, 9):
        d *= 0.5
    return d

def build_flow_distance(df_audio):
    """Build a pairwise distance matrix from key + tempo only."""
    ids = df_audio.index.tolist()
    n = len(ids)

    # normalise tempo to [0,1]
    tempos = df_audio["tempo"].values
    t_min, t_max = tempos.min(), tempos.max()
    t_norm = (tempos - t_min) / (t_max - t_min) if t_max > t_min else np.zeros(n)

    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(i + 1, n):
            kd = key_distance(df_audio.iloc[i]["key"], df_audio.iloc[i]["mode"],
                              df_audio.iloc[j]["key"], df_audio.iloc[j]["mode"])
            td = abs(t_norm[i] - t_norm[j])
            mat[i, j] = mat[j, i] = 0.6 * kd + 0.4 * td  # weight key > tempo
    np.fill_diagonal(mat, np.inf)
    return pd.DataFrame(mat, index=ids, columns=ids)

dist_flow = build_flow_distance(df_audio)
print(f"Flow distance matrix: {dist_flow.shape}")
print(f"  Mean flow dist: {dist_flow.values[np.triu_indices_from(dist_flow.values, k=1)].mean():.4f}")

Flow distance matrix: (427, 427)
  Mean flow dist: 0.2742


In [38]:
# ── Semantic distance (everything EXCEPT key & tempo) ─────────────────
# Combines: audio features (minus key/tempo), lyrics embeddings, image embeddings.
# This is what drives the *grouping* — songs that "feel" similar sit together.

semantic_audio_cols = [c for c in df_audio_norm.columns if c not in ("key", "tempo", "mode")]
df_semantic_audio = df_audio_norm[semantic_audio_cols]

dist_semantic = merge_distance_matrices(
    [euclidean_distance_matrix(df_semantic_audio), dist_lyrics, dist_image],
    weights=[1.0, 1.0, 0.2],
)

# Re-cluster on semantic features only (excludes key/tempo)
semantic_features = merge_features(
    [df_semantic_audio, df_lyrics_norm, df_image_norm],
    weights=[1.0, 1.0, 0.2],
)
print("Semantic features:")
k_semantic = find_optimal_k(semantic_features)
semantic_clusters, semantic_km = kmeans_cluster(semantic_features, n_clusters=k_semantic)

# ── Merge tiny clusters into their nearest neighbour ──────────────────
# Clusters with < min_size songs create jarring short sections.
def merge_small_clusters(clusters, features, min_size=8):
    centroids = {}
    for c in sorted(clusters.unique()):
        centroids[c] = features.loc[clusters == c].mean().values
    merged = clusters.copy()
    for c in sorted(centroids.keys()):
        if (merged == c).sum() < min_size:
            best_target = min(
                (o for o in centroids if o != c and (merged == o).sum() >= min_size),
                key=lambda o: np.linalg.norm(centroids[c] - centroids[o]),
                default=None,
            )
            if best_target is not None:
                merged[merged == c] = best_target
    # re-label to contiguous 0..n
    label_map = {old: new for new, old in enumerate(sorted(merged.unique()))}
    return merged.map(label_map)

semantic_clusters = merge_small_clusters(semantic_clusters, semantic_features, min_size=8)

for c in sorted(semantic_clusters.unique()):
    print(f"  Semantic cluster {c}: {(semantic_clusters == c).sum()} songs")

Semantic features:


  k=2: silhouette=0.2658
  k=3: silhouette=0.2934
  k=4: silhouette=0.2939
  k=5: silhouette=0.2863
  k=6: silhouette=0.2435
  k=7: silhouette=0.0917
  k=8: silhouette=0.0900
  k=9: silhouette=0.0954
  k=10: silhouette=0.0803
  → Best k=4 (silhouette=0.2939)
  Semantic cluster 0: 44 songs
  Semantic cluster 1: 59 songs
  Semantic cluster 2: 72 songs
  Semantic cluster 3: 252 songs


In [39]:
# ── Playlist builder (improved) ───────────────────────────────────────
# Improvements over v1:
#   - 2-opt local refinement within each group to iron out greedy mis-steps
#   - Boundary-aware stitching: try reversed group direction to minimise
#     the flow spike at each cluster transition
#   - Capped variance injection: only inject variance when flow stays low

def order_within_group(song_ids, dist_flow_mat, dist_semantic_mat,
                       variance_every=5, variance_strength=0.25,
                       max_variance_flow=0.15, seed=None):
    """
    Greedy nearest-neighbour on key/tempo flow, with capped variance injection.
    Variance is only injected if the best flow-only candidate has flow < max_variance_flow,
    preventing the variance mechanism from creating spikes.
    """
    if len(song_ids) <= 1:
        return song_ids

    remaining = set(song_ids)
    if seed and seed in remaining:
        current = seed
    else:
        avg_flow = {s: dist_flow_mat.loc[s, list(remaining - {s})].mean() for s in remaining}
        current = min(avg_flow, key=avg_flow.get)
    ordered = [current]
    remaining.remove(current)
    step = 1

    while remaining:
        candidates = list(remaining)
        flow_dists = dist_flow_mat.loc[current, candidates].values
        f_min, f_max = flow_dists.min(), flow_dists.max()
        flow_norm = (flow_dists - f_min) / (f_max - f_min) if f_max > f_min else np.zeros(len(candidates))
        flow_closeness = 1.0 - flow_norm

        inject_variance = (
            step % variance_every == 0
            and len(candidates) > 1
            and flow_dists.min() < max_variance_flow  # only inject when flow is already smooth
        )

        if inject_variance:
            sem_dists = dist_semantic_mat.loc[current, candidates].values
            s_min, s_max = sem_dists.min(), sem_dists.max()
            sem_norm = (sem_dists - s_min) / (s_max - s_min) if s_max > s_min else np.zeros(len(candidates))
            score = (1 - variance_strength) * flow_closeness + variance_strength * sem_norm
            best_idx = np.argmax(score)
        else:
            best_idx = np.argmax(flow_closeness)

        current = candidates[best_idx]
        ordered.append(current)
        remaining.remove(current)
        step += 1

    return ordered


def two_opt_refine(ordered, dist_flow_mat, max_iters=500):
    """
    2-opt local search: iteratively reverse sub-segments if it reduces
    total consecutive flow distance.  Cheap and effective for 30-song segments.
    """
    route = list(ordered)
    n = len(route)
    if n < 4:
        return route

    improved = True
    iters = 0
    while improved and iters < max_iters:
        improved = False
        iters += 1
        for i in range(n - 2):
            for j in range(i + 2, n):
                # current cost of edges touching i..i+1 and j..j+1
                old_cost = dist_flow_mat.loc[route[i], route[i + 1]]
                if j + 1 < n:
                    old_cost += dist_flow_mat.loc[route[j], route[j + 1]]

                # cost after reversing route[i+1..j]
                new_cost = dist_flow_mat.loc[route[i], route[j]]
                if j + 1 < n:
                    new_cost += dist_flow_mat.loc[route[i + 1], route[j + 1]]

                if new_cost < old_cost - 1e-9:
                    route[i + 1:j + 1] = route[i + 1:j + 1][::-1]
                    improved = True
    return route


def order_groups(groups, df_audio):
    """Order cluster groups so transitions between groups are smooth."""
    group_stats = {}
    for gid, song_ids in groups.items():
        sub = df_audio.loc[song_ids]
        avg_fifth_pos = sub["key"].map(FIFTHS_POS).mean()
        avg_tempo_norm = (sub["tempo"] - df_audio["tempo"].min()) / (df_audio["tempo"].max() - df_audio["tempo"].min())
        group_stats[gid] = (avg_fifth_pos / 12.0, avg_tempo_norm.mean())

    remaining = set(groups.keys())
    start = min(remaining, key=lambda g: (group_stats[g][0]**2 + (group_stats[g][1] - 0.5)**2))
    order = [start]
    remaining.remove(start)

    while remaining:
        cur = group_stats[order[-1]]
        best = min(remaining, key=lambda g: (
            (group_stats[g][0] - cur[0])**2 + (group_stats[g][1] - cur[1])**2
        ))
        order.append(best)
        remaining.remove(best)

    return order

def optimize_boundary(group_a, group_b, dist_flow, k=3):
    """
    Try moving up to k songs from the tail of group_a and head of group_b
    to find the pair that minimises the cross-boundary flow distance.
    Returns (group_a, group_b) with the best boundary songs repositioned.
    """
    best_cost = dist_flow.loc[group_a[-1], group_b[0]]
    best_a, best_b = group_a, group_b

    tail_candidates = min(k, len(group_a) - 1)
    head_candidates = min(k, len(group_b) - 1)

    for ti in range(tail_candidates):
        for hi in range(head_candidates):
            # try swapping song at position -(ti+1) to end of A,
            # and song at position hi to start of B
            a_variant = [s for j, s in enumerate(group_a) if j != len(group_a) - 1 - ti]
            a_variant.append(group_a[len(group_a) - 1 - ti])
            b_variant = [group_b[hi]] + [s for j, s in enumerate(group_b) if j != hi]

            cost = dist_flow.loc[a_variant[-1], b_variant[0]]
            if cost < best_cost - 1e-9:
                best_cost = cost
                best_a, best_b = a_variant, b_variant

    return best_a, best_b, best_cost

def or_opt_refine(route, dist_flow, max_iters=300):
    """Relocate single songs to better positions."""
    route = list(route)
    n = len(route)
    improved = True
    iters = 0
    while improved and iters < max_iters:
        improved = False
        iters += 1
        for i in range(1, n - 1):  # don't move endpoints
            # cost of removing route[i]
            old_cost = (dist_flow.loc[route[i-1], route[i]] +
                        dist_flow.loc[route[i], route[i+1]])
            skip_cost = dist_flow.loc[route[i-1], route[i+1]]
            removal_saving = old_cost - skip_cost

            best_gain = 0
            best_j = None
            for j in range(n):
                if j in (i-1, i, i+1):
                    continue
                if j < n - 1:
                    insert_cost = (dist_flow.loc[route[j], route[i]] +
                                   dist_flow.loc[route[i], route[j+1]] -
                                   dist_flow.loc[route[j], route[j+1]])
                else:
                    insert_cost = dist_flow.loc[route[j], route[i]]

                gain = removal_saving - insert_cost
                if gain > best_gain + 1e-9:
                    best_gain = gain
                    best_j = j

            if best_j is not None:
                song = route.pop(i)
                insert_at = best_j if best_j < i else best_j
                route.insert(insert_at + 1, song)
                improved = True
                break  # restart scan
    return route

def build_playlist(semantic_clusters, dist_flow, dist_semantic, df_audio,
                   variance_every=5, variance_strength=0.25):
    """Full playlist: group → order groups → order within → 2-opt → stitch → concatenate."""
    # 1. Group songs by semantic cluster
    groups = {}
    for sid, clust in semantic_clusters.items():
        groups.setdefault(clust, []).append(sid)

    # 2. Order the groups for smooth inter-group transitions
    group_order = order_groups(groups, df_audio)

    # 3. Order songs within each group + 2-opt + or-opt refinement
    ordered_groups = {}
    for idx, gid in enumerate(group_order):
        seed = None
        if idx > 0:
            prev_gid = group_order[idx - 1]
            prev_songs = groups[prev_gid]
            prev_avg_flow = dist_flow.loc[prev_songs, groups[gid]].mean(axis=0)
            seed = prev_avg_flow.idxmin()

        ordered = order_within_group(
            groups[gid], dist_flow, dist_semantic,
            variance_every=variance_every,
            variance_strength=variance_strength,
            seed=seed,
        )
        ordered = two_opt_refine(ordered, dist_flow)
        ordered = or_opt_refine(ordered, dist_flow)  # new pass
        ordered_groups[gid] = ordered

    # 4. Boundary-aware stitching: for each group, try fwd vs reversed
    #    to minimise the flow distance at each cluster boundary.
    #    Also try reversing the first group to get a better global start.
    for first_reversed in [False, True]:
        candidate = {}
        first_gid = group_order[0]
        candidate[first_gid] = (
            list(reversed(ordered_groups[first_gid])) if first_reversed
            else ordered_groups[first_gid]
        )
        for i in range(1, len(group_order)):
            prev_gid = group_order[i - 1]
            curr_gid = group_order[i]
            prev_last = candidate[prev_gid][-1]
            fwd = ordered_groups[curr_gid]
            rev = list(reversed(fwd))
            cost_fwd = dist_flow.loc[prev_last, fwd[0]]
            cost_rev = dist_flow.loc[prev_last, rev[0]]
            candidate[curr_gid] = rev if cost_rev < cost_fwd else fwd

        if first_reversed:
            stitched_rev = candidate
        else:
            stitched_fwd = candidate

    # pick whichever stitching gives lower total boundary cost
    def boundary_cost(stitched):
        cost = 0
        for i in range(1, len(group_order)):
            cost += dist_flow.loc[
                stitched[group_order[i - 1]][-1],
                stitched[group_order[i]][0]
            ]
        return cost

    stitched = (stitched_rev if boundary_cost(stitched_rev) < boundary_cost(stitched_fwd)
                else stitched_fwd)
    
    
    # 4b. Optimize boundary songs between adjacent groups
    for i in range(1, len(group_order)):
        prev_gid = group_order[i - 1]
        curr_gid = group_order[i]
        stitched[prev_gid], stitched[curr_gid], _ = optimize_boundary(
            stitched[prev_gid], stitched[curr_gid], dist_flow, k=3
        )
        

    # 5. Concatenate
    playlist = []
    group_boundaries = []
    for gid in group_order:
        start_pos = len(playlist)
        playlist.extend(stitched[gid])
        group_boundaries.append((gid, start_pos, len(playlist)))

    return playlist, group_boundaries

playlist, boundaries = build_playlist(
    semantic_clusters, dist_flow, dist_semantic, df_audio,
    variance_every=5, variance_strength=0.25,
)

print(f"Playlist: {len(playlist)} songs")
for gid, start, end in boundaries:
    print(f"  Cluster {gid}: positions {start}–{end-1} ({end - start} songs)")

Playlist: 427 songs
  Cluster 2: positions 0–71 (72 songs)
  Cluster 0: positions 72–115 (44 songs)
  Cluster 1: positions 116–174 (59 songs)
  Cluster 3: positions 175–426 (252 songs)


In [40]:
# ── Display playlist with song names + flow stats ─────────────────────
db.connect(reuse_if_open=True)
song_names = {str(s.id): f"{s.artists[0] if s.artists else 'Unknown Artist'} — {s.song_name}" for s in Songs.select()}
db.close()

current_group = None
for i, sid in enumerate(playlist):
    # find which group boundary we're in
    for gid, start, end in boundaries:
        if start <= i < end:
            if gid != current_group:
                current_group = gid
                print(f"\n{'='*60}")
                print(f"  ▸ Cluster {gid}  ({end - start} songs)")
                print(f"{'='*60}")
            break

    key_val = int(df_audio.loc[sid, "key"])
    mode_val = int(df_audio.loc[sid, "mode"])
    tempo_val = df_audio.loc[sid, "tempo"]
    key_names = ["C","C#","D","D#","E","F","F#","G","G#","A","A#","B"]
    mode_str = "maj" if mode_val == 1 else "min"

    # flow distance from previous song
    if i > 0:
        flow_d = dist_flow.loc[playlist[i-1], sid]
        arrow = f"  (Δflow={flow_d:.3f})"
    else:
        arrow = ""

    name = song_names.get(sid, sid)
    print(f"  {i+1:3d}. {name:<45s}  {key_names[key_val]} {mode_str}  {tempo_val:6.1f} BPM{arrow}")


  ▸ Cluster 2  (72 songs)
    1. Søte & Rare — Kun i kveld                      C# min   136.0 BPM
    2. Søte & Rare — Bender                           A min   152.0 BPM  (Δflow=0.458)
    3. Søte & Rare — Heartless                        A min   152.0 BPM  (Δflow=0.000)
    4. Roc Boyz — Backstreetboyz                      A min   143.6 BPM  (Δflow=0.031)
    5. Roc Boyz — Disc Jockey                         A min   143.6 BPM  (Δflow=0.000)
    6. Søte & Rare — Hvitt Gull (Kilroy)              A min   143.6 BPM  (Δflow=0.000)
    7. Fjellrev — Liten Tiss (Rodeo)                  A min   143.6 BPM  (Δflow=0.000)
    8. Problembarn — Vi Jagger                        A min   143.6 BPM  (Δflow=0.000)
    9. Fjellrev — Distriktet                          A min   143.6 BPM  (Δflow=0.000)
   10. Problembarn — Madchester                       A min   143.6 BPM  (Δflow=0.000)
   11. Problembarn — After Eight                      A min   143.6 BPM  (Δflow=0.000)
   12. Roc Boyz — Matilda     

In [41]:
# ── Visualise flow quality ────────────────────────────────────────────
flow_deltas = [dist_flow.loc[playlist[i], playlist[i+1]] for i in range(len(playlist)-1)]

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), gridspec_kw={"height_ratios": [2, 1]})

# Top: flow distance between consecutive songs
colors = []
for i in range(len(flow_deltas)):
    for gid, start, end in boundaries:
        if start <= i < end:
            colors.append(f"C{gid}")
            break
ax1.bar(range(len(flow_deltas)), flow_deltas, color=colors, alpha=0.8, width=1.0)
for gid, start, end in boundaries:
    if start > 0:
        ax1.axvline(start - 0.5, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
    ax1.text((start + end) / 2 - 0.5, max(flow_deltas) * 0.95, f"C{gid}",
             ha="center", fontsize=9, fontweight="bold")
ax1.set_ylabel("Flow distance (key+tempo)")
ax1.set_title("Consecutive Song Flow Distance Across Playlist")
ax1.set_xlim(-0.5, len(flow_deltas) - 0.5)

# Bottom: tempo + key progression
tempos = [df_audio.loc[sid, "tempo"] for sid in playlist]
keys = [FIFTHS_POS[int(df_audio.loc[sid, "key"])] for sid in playlist]
ax2.plot(tempos, label="Tempo (BPM)", color="steelblue", linewidth=1.2)
ax2_twin = ax2.twinx()
ax2_twin.plot(keys, label="Key (5ths pos)", color="coral", linewidth=1.2, alpha=0.8)
ax2.set_xlabel("Playlist position")
ax2.set_ylabel("Tempo (BPM)", color="steelblue")
ax2_twin.set_ylabel("Key (circle of 5ths)", color="coral")
for gid, start, end in boundaries:
    if start > 0:
        ax2.axvline(start, color="black", linewidth=1.5, linestyle="--", alpha=0.6)
ax2.set_title("Tempo & Key Progression")

fig.tight_layout()
fig.savefig("playlist_flow.png", dpi=150, bbox_inches="tight")

mean_flow = np.mean(flow_deltas)
max_flow = np.max(flow_deltas)
spikes = sum(1 for d in flow_deltas if d > 0.3)
print(f"Mean consecutive flow distance: {mean_flow:.4f}")
print(f"Max consecutive flow distance:  {max_flow:.4f}")
print(f"Transitions > 0.3:             {spikes} / {len(flow_deltas)}")

Mean consecutive flow distance: 0.0178
Max consecutive flow distance:  0.4583
Transitions > 0.3:             1 / 426
